# Walmart Store Sales Forecasting — LSTM vs GRU

**Business Problem:** Walmart needs to forecast weekly sales for each store and department to optimize inventory and staffing, especially for promotional seasons like Super Bowl, Labor Day, Thanksgiving, and Christmas.

**Approach:** Global LSTM/GRU model — trains a single unified model on all stores and departments to learn cross-store sales patterns.

**Evaluation Metrics:** MAE, RMSE, **WMAE (holiday weight ×5)**

## 1. Environment Setup & Data Extraction

In [ ]:
# Upload data to Colab (only needs to run once)
from google.colab import files
import shutil, os

print("Please select the local walmart-recruiting-store-sales-forecasting.zip file to upload...")
uploaded = files.upload()

os.makedirs('project', exist_ok=True)
fname = list(uploaded.keys())[0]
shutil.move(fname, 'project/walmart-recruiting-store-sales-forecasting.zip')
print("Upload complete! File saved to project/ directory")

In [ ]:
import zipfile, os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

warnings.filterwarnings('ignore')
print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

In [ ]:
zip_path = 'project/walmart-recruiting-store-sales-forecasting.zip'

if not os.path.exists('project/train.csv'):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall('project/')
    print('Extraction complete')
else:
    print('Data already exists, skipping extraction')

print(os.listdir('project/'))

In [ ]:
zip_path = 'project/walmart-recruiting-store-sales-forecasting.zip'

# Extract outer zip
if not os.path.exists('project/train.csv') and not os.path.exists('project/train.csv.zip'):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall('project/')
    print('Outer archive extracted')

# Extract inner .csv.zip (Kaggle nested compression)
for inner_zip in ['train.csv.zip', 'features.csv.zip', 'test.csv.zip', 'sampleSubmission.csv.zip']:
    inner_path = f'project/{inner_zip}'
    csv_path   = inner_path.replace('.zip', '')
    if os.path.exists(inner_path) and not os.path.exists(csv_path):
        with zipfile.ZipFile(inner_path) as z:
            z.extractall('project/')
        print(f'Extracted: {inner_zip}')

print('\nExtraction complete, project/ directory contents:')
print([f for f in os.listdir('project/') if f.endswith('.csv')])


## 2. Data Loading and Merging

| File | Content | Rows |
|------|------|------|
| `train.csv` | Historical weekly sales data (2010-02-05 to 2012-10-26) | 421,570 |
| `features.csv` | External features (temperature, fuel price, markdowns, CPI, unemployment) | 8,190 |
| `stores.csv` | Store type (A/B/C) and size | 45 |

In [ ]:
train    = pd.read_csv('project/train.csv')
features = pd.read_csv('project/features.csv')
stores   = pd.read_csv('project/stores.csv')

print('train shape:   ', train.shape)
print('features shape:', features.shape)
print('stores shape:  ', stores.shape)
train.head()

In [ ]:
df = train.merge(features, on=['Store', 'Date', 'IsHoliday'], how='left') \
          .merge(stores,   on='Store', how='left')

df['Date'] = pd.to_datetime(df['Date'])
df.sort_values(['Store', 'Dept', 'Date'], inplace=True)
df.reset_index(drop=True, inplace=True)

print('Merged shape:', df.shape)
df.head()

## 3. Exploratory Data Analysis

In [ ]:
print('Date range:', df['Date'].min(), '→', df['Date'].max())
print('Number of stores:', df['Store'].nunique())
print('Number of departments:', df['Dept'].nunique())
print('Store+Dept combinations:', df.groupby(['Store','Dept']).ngroups)
print('Negative sales records:', (df['Weekly_Sales'] < 0).sum())
print()
for col in ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']:
    print(f'{col} missing rate: {df[col].isna().mean()*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

monthly = df.groupby(df['Date'].dt.month)['Weekly_Sales'].mean()
axes[0].bar(monthly.index, monthly.values, color='steelblue')
axes[0].set_title('Monthly Average Weekly Sales (Seasonality)')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Average Sales (USD)')
axes[0].xaxis.set_major_locator(ticker.MultipleLocator(1))

holiday_sales = df.groupby('IsHoliday')['Weekly_Sales'].mean()
axes[1].bar(['Non-Holiday', 'Holiday'], holiday_sales.values, color=['steelblue', 'coral'])
axes[1].set_title('Holiday vs Non-Holiday Average Sales')
axes[1].set_ylabel('Average Sales (USD)')

plt.tight_layout()
plt.show()
print(f'Holiday sales are {(holiday_sales[1]/holiday_sales[0]-1)*100:.1f}% higher than non-holiday sales')


## 4. Feature Engineering

Total of **17 input features**:

| Category | Features |
|------|------|
| Time | week, month, year |
| Holiday | IsHoliday, super_bowl, labor_day, thanksgiving, christmas |
| Markdown | MarkDown1~5 (fill 0 for missing, indicating no promotions) |
| Macro-economic | Temperature, Fuel_Price, CPI, Unemployment |

In [ ]:
df['week']  = df['Date'].dt.isocalendar().week.astype(int)
df['month'] = df['Date'].dt.month
df['year']  = df['Date'].dt.year

df['super_bowl']   = ((df['month'] == 2)  & df['IsHoliday']).astype(int)
df['labor_day']    = ((df['month'] == 9)  & df['IsHoliday']).astype(int)
df['thanksgiving'] = ((df['month'] == 11) & df['IsHoliday']).astype(int)
df['christmas']    = ((df['month'] == 12) & df['IsHoliday']).astype(int)
df['IsHoliday']    = df['IsHoliday'].astype(int)

for col in ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']:
    df[col] = df[col].fillna(0)

df['Type'] = df['Type'].map({'A': 0, 'B': 1, 'C': 2})

FEATURE_COLS = [
    'week', 'month', 'year',
    'IsHoliday', 'super_bowl', 'labor_day', 'thanksgiving', 'christmas',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment'
]
TARGET_COL = 'Weekly_Sales'

print(f'Number of features: {len(FEATURE_COLS)}')
df[FEATURE_COLS].describe().round(2)

## 5. Sequence Construction & Dataset Split

- **Filter**: Retain Store+Dept combinations with ≥ 100 time steps
- `SEQ_LEN = 26` (use past 6 months as input)
- `PRED_LEN = 4` (forecast the next 4 weeks)
- **Chronological split**: Train 60% / Validation 20% / Test 20% (no random shuffling)

In [ ]:
SEQ_LEN = 12  # Shorter window to increase available samples
PRED_LEN = 4
MIN_LEN = 60

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

# Fit scalers
train_mask = df['Date'] <= '2011-10-31'
feature_scaler.fit(df.loc[train_mask, FEATURE_COLS])
target_scaler.fit(df.loc[train_mask, [TARGET_COL]])

X_train_list, y_train_list = [], []
X_val_list, y_val_list = [], []
X_test_list, y_test_list = [], []
is_holiday_test_list = []

def make_windows(X, y, hol, start, end):
    xs, ys, hs = [], [], []
    # Fix index logic to ensure windows can be generated even with fewer data points
    for i in range(start, end - SEQ_LEN - PRED_LEN + 1):
        xs.append(X[i : i + SEQ_LEN])
        ys.append(y[i + SEQ_LEN : i + SEQ_LEN + PRED_LEN])
        hs.append(hol[i + SEQ_LEN : i + SEQ_LEN + PRED_LEN])
    return xs, ys, hs

kept = 0
for (store, dept), grp in df.groupby(['Store', 'Dept']):
    grp = grp.sort_values('Date').reset_index(drop=True)
    if len(grp) < MIN_LEN: continue
    kept += 1

    X_sc = feature_scaler.transform(grp[FEATURE_COLS])
    y_sc = target_scaler.transform(grp[[TARGET_COL]]).flatten()
    is_hol = grp['IsHoliday'].values

    n = len(grp)
    cut1 = int(n * 0.7)  # Training set 70%
    cut2 = int(n * 0.85) # Validation set 15%

    # Use sliding windows to generate datasets
    xtr, ytr, _ = make_windows(X_sc, y_sc, is_hol, 0, cut1)
    xv, yv, _ = make_windows(X_sc, y_sc, is_hol, cut1 - SEQ_LEN, cut2)
    xte, yte, hte = make_windows(X_sc, y_sc, is_hol, cut2 - SEQ_LEN, n)

    X_train_list += xtr; y_train_list += ytr
    X_val_list += xv; y_val_list += yv
    X_test_list += xte; y_test_list += yte
    is_holiday_test_list += hte

# Convert to Numpy arrays and check dimensions
X_train, y_train = np.array(X_train_list, dtype='float32'), np.array(y_train_list, dtype='float32')
X_val, y_val = np.array(X_val_list, dtype='float32'), np.array(y_val_list, dtype='float32')
X_test, y_test = np.array(X_test_list, dtype='float32'), np.array(y_test_list, dtype='float32')
is_holiday_test = np.array(is_holiday_test_list, dtype='float32')

print(f'Successfully processed combinations: {kept}')
print(f'X_train shape: {X_train.shape}')
print(f'X_val shape: {X_val.shape}')
print(f'X_test shape: {X_test.shape}')

## 6. Evaluation Function

**WMAE** (Weighted Mean Absolute Error) is the primary evaluation metric, with holiday weight ×5:

$$WMAE = \frac{\sum w_i |y_i - \hat{y}_i|}{\sum w_i}, \quad w_i = \begin{cases} 5 & \text{holiday} \\ 1 & \text{otherwise} \end{cases}$$

In [ ]:
def evaluate(y_true_sc, y_pred_sc, is_hol_flags, scaler, label=''):
    y_true  = scaler.inverse_transform(y_true_sc.reshape(-1, 1)).flatten()
    y_pred  = scaler.inverse_transform(y_pred_sc.reshape(-1, 1)).flatten()
    weights = np.where(is_hol_flags.flatten() > 0, 5.0, 1.0)

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    wmae = (weights * np.abs(y_true - y_pred)).sum() / weights.sum()

    if label:
        print(f'[{label}]  MAE: ${mae:>10,.0f}  |  RMSE: ${rmse:>10,.0f}  |  WMAE: ${wmae:>10,.0f}')
    return mae, rmse, wmae, y_true, y_pred

print('Evaluation function defined')

## 7. LSTM Model

```
Input(26 weeks, 17 features)
  → LSTM(64, return_sequences=True) → Dropout(0.2)
  → LSTM(32, return_sequences=False) → Dropout(0.2)
  → Dense(4)  ← predict next 4 weeks of sales
```

In [ ]:
def build_model(model_type='LSTM'):
    RNNLayer = LSTM if model_type == 'LSTM' else GRU
    model = Sequential([
        RNNLayer(64, return_sequences=True,  input_shape=(SEQ_LEN, len(FEATURE_COLS))),
        Dropout(0.2),
        RNNLayer(32, return_sequences=False),
        Dropout(0.2),
        Dense(PRED_LEN)
    ], name=f'{model_type}_model')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.001),
        loss='mse',
        metrics=['mae']
    )
    return model

lstm_model = build_model('LSTM')
lstm_model.summary()

In [ ]:
# Ensure input dimensions match X_train
lstm_model = build_model('LSTM')

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(patience=3, factor=0.5, min_lr=1e-5, verbose=1)
]

print("Starting LSTM model training...")
t0 = time.time()
lstm_history = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)
lstm_time = time.time() - t0
print(f'\nLSTM training complete, elapsed: {lstm_time:.1f} seconds')

In [ ]:
SEQ_LEN  = 12
PRED_LEN = 4
MIN_LEN  = 60

feature_scaler = StandardScaler()
target_scaler  = StandardScaler()

train_mask = df['Date'] <= '2011-10-31'
feature_scaler.fit(df.loc[train_mask, FEATURE_COLS])
target_scaler.fit(df.loc[train_mask, [TARGET_COL]])

X_train_list, y_train_list = [], []
X_val_list,   y_val_list   = [], []
X_test_list,  y_test_list  = [], []
is_holiday_test_list = []

def make_windows(X, y, hol, start, end):
    xs, ys, hs = [], [], []
    # Ensure the loop range is valid
    for i in range(start, end - SEQ_LEN - PRED_LEN + 1):
        xs.append(X[i : i + SEQ_LEN])
        ys.append(y[i + SEQ_LEN : i + SEQ_LEN + PRED_LEN])
        hs.append(hol[i + SEQ_LEN : i + SEQ_LEN + PRED_LEN])
    return xs, ys, hs

kept = 0
for (store, dept), grp in df.groupby(['Store', 'Dept']):
    grp = grp.sort_values('Date').reset_index(drop=True)
    if len(grp) < MIN_LEN:
        continue
    kept += 1

    X_sc   = feature_scaler.transform(grp[FEATURE_COLS])
    y_sc   = target_scaler.transform(grp[[TARGET_COL]]).flatten()
    is_hol = grp['IsHoliday'].values

    n    = len(grp)
    cut1 = int(n * 0.7) # Adjust ratio to ensure validation set has data
    cut2 = int(n * 0.85)

    xtr, ytr, _   = make_windows(X_sc, y_sc, is_hol, 0,    cut1)
    xv,  yv,  _   = make_windows(X_sc, y_sc, is_hol, cut1 - SEQ_LEN, cut2) # Overlapping sliding window ensures continuity
    xte, yte, hte = make_windows(X_sc, y_sc, is_hol, cut2 - SEQ_LEN, n)

    X_train_list += xtr;  y_train_list += ytr
    X_val_list   += xv;   y_val_list   += yv
    X_test_list  += xte;  y_test_list  += yte
    is_holiday_test_list += hte

def safe_stack(lst):
    return np.array(lst, dtype=np.float32)

X_train = safe_stack(X_train_list)
y_train = safe_stack(y_train_list)
X_val   = safe_stack(X_val_list)
y_val   = safe_stack(y_val_list)
X_test  = safe_stack(X_test_list)
y_test  = safe_stack(y_test_list)
is_holiday_test = safe_stack(is_holiday_test_list)

print(f'Retained combinations: {kept}')
print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_val:   {X_val.shape}    y_val:   {y_val.shape}')
print(f'X_test:  {X_test.shape}   y_test:  {y_test.shape}')

In [ ]:
lstm_pred = lstm_model.predict(X_test, batch_size=512, verbose=0)
lstm_mae, lstm_rmse, lstm_wmae, y_true_orig, lstm_pred_orig = evaluate(
    y_test, lstm_pred, is_holiday_test, target_scaler, 'LSTM Test Set'
)

## 8. GRU Comparison Model

Identical architecture to LSTM, with LSTM layers replaced by GRU.
- GRU has fewer parameters (no separate output gate), trains faster
- Compare speed vs. accuracy trade-offs between both models on the same task

In [ ]:
gru_model = build_model('GRU')
gru_model.summary()

In [ ]:
t0 = time.time()
gru_history = gru_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=256,
    callbacks=[
        EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(patience=3, factor=0.5, min_lr=1e-5, verbose=1)
    ],
    verbose=1
)
gru_time = time.time() - t0
print(f'\nGRU training time: {gru_time:.1f} seconds')

In [ ]:
gru_pred = gru_model.predict(X_test, batch_size=512, verbose=0)
gru_mae, gru_rmse, gru_wmae, _, gru_pred_orig = evaluate(
    y_test, gru_pred, is_holiday_test, target_scaler, 'GRU  Test Set'
)

print(f'\nGRU trains {(lstm_time - gru_time)/lstm_time*100:.1f}% faster than LSTM')

## 9. Visualization of Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# (1) LSTM Loss Curve
ax = axes[0, 0]
ax.plot(lstm_history.history['loss'], label='Train Loss')
ax.plot(lstm_history.history['val_loss'], label='Val Loss')
ax.set_title('LSTM Loss Curve')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.legend()

# (2) GRU Loss Curve
ax = axes[0, 1]
ax.plot(gru_history.history['loss'], label='Train Loss', color='orange')
ax.plot(gru_history.history['val_loss'], label='Val Loss', color='red')
ax.set_title('GRU Loss Curve')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.legend()

# (3) Prediction vs Actual
ax = axes[1, 0]
n_show = min(200, len(y_true_orig))
ax.plot(y_true_orig[:n_show],    label='Actual Sales', alpha=0.8)
ax.plot(lstm_pred_orig[:n_show], label='LSTM Pred',  alpha=0.7, linestyle='--')
ax.plot(gru_pred_orig[:n_show],  label='GRU Pred',   alpha=0.7, linestyle=':')
ax.set_title('Test Set Prediction Comparison (First 200 samples)')
ax.set_xlabel('Sample Index')
ax.set_ylabel('Sales (USD)')
ax.legend()

# (4) Metrics Comparison Bar Chart
ax = axes[1, 1]
metrics   = ['MAE', 'RMSE', 'WMAE']
lstm_vals = [lstm_mae, lstm_rmse, lstm_wmae]
gru_vals  = [gru_mae,  gru_rmse,  gru_wmae]

x = np.arange(len(metrics))
w = 0.35
b1 = ax.bar(x - w/2, lstm_vals, w, label='LSTM', color='steelblue')
b2 = ax.bar(x + w/2, gru_vals,  w, label='GRU',  color='coral')
ax.set_title('LSTM vs GRU Metrics Comparison')
ax.set_ylabel('Error (USD)')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
            f'${bar.get_height():,.0f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Walmart Sales Forecasting — LSTM vs GRU', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Results Summary

In [ ]:
summary = pd.DataFrame({
    'Model':         ['LSTM', 'GRU'],
    'MAE ($)':     [f'{lstm_mae:,.0f}',  f'{gru_mae:,.0f}'],
    'RMSE ($)':    [f'{lstm_rmse:,.0f}', f'{gru_rmse:,.0f}'],
    'WMAE ($)':    [f'{lstm_wmae:,.0f}', f'{gru_wmae:,.0f}'],
    'Training Time (s)': [f'{lstm_time:.1f}', f'{gru_time:.1f}']
})

print('=== Final Evaluation Results ===')
display(summary)

winner = 'GRU' if gru_wmae < lstm_wmae else 'LSTM'
faster = 'GRU' if gru_time < lstm_time else 'LSTM'
print(f'\nBest accuracy (WMAE): {winner}')
print(f'Faster training: {faster}')

## 11. Key Findings & Future Improvements

### Design Decision Review

| Decision | Reason |
|------|------|
| **Global Model** | Training 3,331 separate Store+Dept models is infeasible; global model learns cross-store patterns |
| **WMAE as Key Metric** | Holiday sales are more critical for inventory decisions; ×5 weight makes the model focus on holiday accuracy |
| **MarkDown Fill with 0** | 50%+ missing rate; filling 0 means "no promotion", retains more information than dropna |
| **LSTM vs GRU** | GRU has fewer params and trains faster; LSTM theoretically handles longer dependencies; experiment quantifies the difference |

### Future Improvement Directions

1. **Richer Store Features**: Add static features per Store+Dept such as historical mean and trend slope
2. **Transformer Models**: Temporal Fusion Transformer (TFT) excels in retail forecasting
3. **Rolling Forecast**: Feed predictions back as input for longer-term forecasting

### 12. Advanced Optimization Baseline: Refined GRU (Base)
(Early Stopping) strategy to build a stable baseline GRU model.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
import time

# 1. Configure a more sensitive early stopping strategy
custom_early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    min_delta=0.0001,
    restore_best_weights=True,
    verbose=1
)

# 2. Build and train Refined GRU
refined_gru_model = build_model('GRU')

print("Starting Refined GRU (Base) training...")
t0 = time.time()
refined_history = refined_gru_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=256,
    callbacks=[custom_early_stopping],
    verbose=1
)
refined_time = time.time() - t0

# 3. Evaluate metrics
refined_pred = refined_gru_model.predict(X_test, batch_size=512, verbose=0)
refined_mae, refined_rmse, refined_wmae, _, _ = evaluate(
    y_test, refined_pred, is_holiday_test, target_scaler, 'Refined GRU'
)

print(f'\nTraining complete! WMAE: ${refined_wmae:,.0f}')

In [ ]:
import matplotlib.pyplot as plt

# Inverse-transform Refined GRU predictions (manually if not saved in evaluate)
y_refined_true = target_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
y_refined_pred = target_scaler.inverse_transform(refined_pred.reshape(-1, 1)).flatten()

plt.figure(figsize=(15, 6))

# Select the first 300 data points for a clearer view
n_points = 300
plt.plot(y_refined_true[:n_points], label='Actual Sales', color='royalblue', linewidth=2, alpha=0.8)
plt.plot(y_refined_pred[:n_points], label='Refined GRU Prediction', color='crimson', linestyle='--', linewidth=1.5, alpha=0.9)

plt.title('Refined GRU Model: Actual vs Predicted Sales (Test Set Sample)', fontsize=14)
plt.xlabel('Sample Index (Test Set)', fontsize=12)
plt.ylabel('Weekly Sales (USD)', fontsize=12)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)

# Annotate WMAE result as reference
plt.text(0.02, 0.95, f'Refined GRU WMAE: ${refined_wmae:,.0f}', transform=plt.gca().transAxes,
         fontsize=12, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

plt.tight_layout()
plt.show()

### 13. Final Model Performance Comparison

We summarize the following three model stages:
1.  **LSTM**: Initial Long Short-Term Memory baseline.
2.  **GRU (Initial)**: Initial Gated Recurrent Unit baseline.
3.  **Refined GRU (Base)**: Stabilized version with `EarlyStopping`.


In [ ]:
import pandas as pd

# Ensure all variables are computed (display '-' if missing)
def get_val(var_name):
    return globals().get(var_name, None)

# Comparison data after removing Heavy Dropout GRU
comparison_data = {
    'Metric': ['MAE ($)', 'RMSE ($)', 'WMAE ($)'],
    'LSTM (Initial)': [get_val('lstm_mae'), get_val('lstm_rmse'), get_val('lstm_wmae')],
    'GRU (Initial)': [get_val('gru_mae'), get_val('gru_rmse'), get_val('gru_wmae')],
    'Refined GRU': [get_val('refined_mae'), get_val('refined_rmse'), get_val('refined_wmae')]
}

final_df = pd.DataFrame(comparison_data)

# Use Styler to format output
print("=== Walmart Sales Forecasting: Core Model Performance Comparison ===")
display(final_df.style.format({
    'LSTM (Initial)': '${:,.0f}',
    'GRU (Initial)': '${:,.0f}',
    'Refined GRU': '${:,.0f}'
}).highlight_min(axis=1, color='lightgreen', subset=pd.IndexSlice[:, final_df.columns[1:]]))

# Automatically draw conclusions
best_wmae_val = final_df.iloc[2, 1:].min()
best_model_name = final_df.columns[1:][final_df.iloc[2, 1:].values.argmin()]
print(f"\n🏆 Best overall model: {best_model_name} (WMAE: ${best_wmae_val:,.0f})")

# Free memory for Heavy Dropout model (optional)
if 'heavy_model' in locals():
    del heavy_model
    print("\nHeavy Dropout model object cleaned up.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create a 1x3 large figure layout
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# 1. Loss Curve (training process)
axes[0].plot(refined_history.history['loss'], label='Train Loss', lw=2)
axes[0].plot(refined_history.history['val_loss'], label='Val Loss', lw=2)
axes[0].set_title('Refined GRU: Loss Curve', fontsize=14)
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Test Prediction Comparison (sample predictions)
# Already stored inverse-transformed values in y_refined_true and y_refined_pred
n_show = 200
axes[1].plot(y_refined_true[:n_show], label='Actual Sales', color='royalblue', alpha=0.7)
axes[1].plot(y_refined_pred[:n_show], label='Predicted Sales', color='crimson', linestyle='--', alpha=0.8)
axes[1].set_title(f'Prediction vs Actual (First {n_show} samples)', fontsize=14)
axes[1].set_xlabel('Sample Index')
axes[1].set_ylabel('Sales (USD)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Metrics Comparison (bar chart)
metrics_names = ['MAE', 'RMSE', 'WMAE']
metrics_values = [refined_mae, refined_rmse, refined_wmae]
colors = ['#66b3ff', '#99ff99', '#ffcc99']

bars = axes[2].bar(metrics_names, metrics_values, color=colors, edgecolor='black', alpha=0.8)
axes[2].set_title('Refined GRU: Performance Metrics', fontsize=14)
axes[2].set_ylabel('Error Value (USD)')

# Annotate values above bar chart
for bar in bars:
    height = bar.get_height()
    axes[2].text(bar.get_x() + bar.get_width()/2., height + 500,
                f'${height:,.0f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Define export file name
comparison_output_path = 'model_comparison_results.csv'

# Save comparison summary table as CSV
final_df.to_csv(comparison_output_path, index=False, encoding='utf-8-sig')

print(f"Summary table saved to: {comparison_output_path}")
# Display table preview
display(final_df)

### 💾 Save Results to Local Computer
Run the cell below to download the performance evaluation CSV file and the trained Refined GRU model file.

In [ ]:
from google.colab import files
import os

# 1. Save Refined GRU model to file
model_filename = 'refined_gru_walmart_model.h5'
refined_gru_model.save(model_filename)
print(f"Model saved as: {model_filename}")

# 2. Files to download
files_to_download = [
    'model_comparison_results.csv',
    model_filename
]

# 3. Trigger browser to download files one by one
for file_path in files_to_download:
    if os.path.exists(file_path):
        print(f"Triggering download: {file_path}")
        files.download(file_path)
    else:
        print(f"Error: File not found {file_path}")